The most important and fundamental process of training a model is to adjust its weights according to the loss based on ground truth and its predictions. So after getting the models predictions and calculating the loss, the loss value can be optimized (minimized) by adjusting the weights of the model.

In [3]:
import numpy as np

## 1. Random Search

In [ ]:
#X_train is the data where each column is an example (e.g. 3073 x 50,000)
#Y_train are the labels (e.g. 1D array of 50,000)
#L evaluates the loss function
best_loss = float("inf")
for num in range(1000):
    W = np.random.randn(10, 3073)
    loss = L(X_train, Y_train, W)
    if loss < bestloss:
        bestloss = loss
        bestW = w
    

## 2. Random local Search (Hill Climbing)

In [ ]:
W = np.random.randn(10, 3073) * 0.001 # generate random starting W
bestloss = float("inf")

for i in range(1000):
  step_size = 0.0001
  Wtry = W + np.random.randn(10, 3073) * step_size
  loss = L(Xtr_cols, Ytr, Wtry)
  if loss < bestloss:
    W = Wtry
    bestloss = loss

## 3. Gradient Descent

It is not necessary to search arbitrarily for a good descent direction; the optimal direction can be computed analytically as the direction of steepest descent.

The gradient of a function at a given point specifies the direction of steepest increase. Consequently, the negative gradient indicates the direction of steepest decrease. However, the gradient does not determine the magnitude of the update; it only provides directional information.

The step size (learning rate) governs how far one moves along this direction in each update. It is a critical hyperparameter in neural network training. If the step size is too large, the update may overshoot the minimum, potentially increasing the loss rather than decreasing it.

### 3.1 Computing the gradient

$$
\frac{\partial f(\mathbf{x})}{\partial x_i} \approx \frac{f(\mathbf{x} + h \mathbf{e}_i) - f(\mathbf{x} - h \mathbf{e}_i)}{2h}
$$

In [ ]:
def eval_numerical_gradient(f, x):
    """
    Compute numerical gradient of f at x using central differences.

    - f: function mapping R^n -> R
    - x: numpy array at which to evaluate gradient
    """
    grad = np.zeros_like(x, dtype=float)
    h = 1e-5

    for ix in np.ndindex(x.shape):
        old_value = x[ix]

        x[ix] = old_value + h
        fxph = f(x)

        x[ix] = old_value - h
        fxmh = f(x)

        x[ix] = old_value  # restore original value

        grad[ix] = (fxph - fxmh) / (2 * h)

    return grad

Numerical gradients are approximate, very slow to evaluate, but easy to write. Analytic gradients (using Calculus) are exact and fast, but highly error-prone to code. Because of this reason it is good to use analytic gradient but check the implementation with the numerical gradient (**gradient check**)

### 3.2 Vanilla Gradient Descent

In [ ]:
while True:
  weights_grad = evaluate_gradient(loss_fun, data, weights)
  weights += - step_size * weights_grad # perform parameter update

>the simple loop above is at the core of all neural network libraries, most common way to optimize neural network loss functions and it represents the core idea of following the gradient until we are happy with the result

Loading the whole dataset into the memory for only one iteration of the weight update is not efficient

### 3.3 Mini-Batch Gradient Descent

The primary reason for using this to improve both computational efficiency and convergence rate of the training process, by processing smaller subsets of data (batches) the weights can be updated more frequently than vanilla (batch) gradient descent

In [ ]:
while True:
  data_batch = sample_training_data(data, 256) # sample 256 examples
  weights_grad = evaluate_gradient(loss_fun, data_batch, weights)
  weights += - step_size * weights_grad # perform parameter update

### 3.4 Stochastic Gradient Descent
The gradient is calculated for each training example (or a small subset of training examples) rather than the entire dataset. It reduces the computation burden but also makes the learning process noisy

In [ ]:
while True:
  sample = sample_training_data(data, 1) # sample 1 example
  weights_grad = evaluate_gradient(loss_fun, sample, weights)
  weights += - step_size * weights_grad # perform parameter update

## 4. Optimizers for Gradient Descent

Because vanilla gradient descent can be improved by using dynamic learning rates and incorporating memory of previous descent directions, several optimization approaches have been developed.

### 4.1 Momentum

This helps in gradient descent prevent slow, zig-zag movement and makes optimization faster and smoother. Instead of using only the current gradient, momentum also remembers previous updates.

In [8]:
import numpy as np

W = np.random.randn(3, 1)   
V = np.zeros_like(W)       

learning_rate = 0.01
beta = 0.9               

#dummy gradient from backprop
dW = np.array([[0.2],
               [-0.1],
               [0.05]])

#momentum update
V = beta * V - learning_rate * dW
W = W + V


### 4.2 AdaGrad (Adaptive Gradient Descent)
improves standard SGD by diagonally scaling the gradient, so it applies an element-wise scaling of the gradient based on the historical sum of squares in each dimension. it is made possible by maintaining a cache variable that continually accumulates the square of the gradients for each parameter, the update is then calculated by dividing the global learning rate by the square root of the historical gradient sum (adapts the steep size for every single parameter, it scales down the learning rate for dimensions that frequently experience high gradients, while maintaining larger step sizes for infrequent dimensions)

⇒ because the algorithm continuously adds squared positive gradients to the cache, its value monotonically grows and the step size continuously shrinks, which can eventually stop the learning process completely even before reaching absolute minimum ⇒ why newer algorithms like RMSProb and Adam were invented

+: no need to tune learning rate as much, sparse features, scales updates
-:learning rate keeps shrinking over time, can lead to vanishing gradient and stop learning

In [ ]:
w = np.array([8.0])

learning_rate = 1.0
epsilon = 1e-7
iterations = 20

G = np.zeros_like(w)

for t in range(iterations):
    # gradient of f(w) = w^2
    grad = 2 * w

    G += grad ** 2

    #adagrad parameter update
    w = w - (learning_rate / (np.sqrt(G) + epsilon)) * grad

    print(f"Step {t+1}: w = {w}, grad = {grad}, G = {G}")

Step 1: w = [7.00000001], grad = [16.], G = [256.]
Step 2: w = [6.3414954], grad = [14.00000001], G = [452.00000035]
Step 3: w = [5.82917499], grad = [12.6829908], G = [612.85825604]
Step 4: w = [5.40312427], grad = [11.65834999], G = [748.77538049]
Step 5: w = [5.03581763], grad = [10.80624854], G = [865.55038795]
Step 6: w = [4.71193371], grad = [10.07163527], G = [966.98822494]
Step 7: w = [4.42190643], grad = [9.42386742], G = [1055.79750201]
Step 8: w = [4.15928449], grad = [8.84381285], G = [1134.01052781]
Step 9: w = [3.91946854], grad = [8.31856898], G = [1203.2091177]
Step 10: w = [3.69903858], grad = [7.83893708], G = [1264.65805227]
Step 11: w = [3.49536611], grad = [7.39807716], G = [1319.38959797]
Step 12: w = [3.30637631], grad = [6.99073223], G = [1368.25993502]
Step 13: w = [3.13039494], grad = [6.61275263], G = [1411.98843235]
Step 14: w = [2.96604574], grad = [6.26078988], G = [1451.18592227]
Step 15: w = [2.81217942], grad = [5.93209148], G = [1486.37563155]
Step 16:

### 4.3 RMSProp (Root Mean Square Propagation)


designed to address the fatal flaw of its predecessor, AdaGrad the monotonically increasing cache of all past squared gradients, causing the model to stop learning. RMSProp updates the weights similarly to AdaGrad by dividing the global learning rate by the square root of stabilized cache of squared gradients. This prevents the learning rate from shrinking too aggressively, allowing the model to continue learning efficiently throughout training.

In [10]:
import numpy as np

w = np.array([8.0])

learning_rate = 0.1
beta = 0.9
epsilon = 1e-8
iterations = 20

Eg2 = np.zeros_like(w)

for t in range(iterations):
    grad = 2 * w  # derivative of f(w)=w^2

    # moving average of squared gradients
    Eg2 = beta * Eg2 + (1to  - beta) * (grad ** 2)

    # RMSProp update
    w = w - (learning_rate / (np.sqrt(Eg2) + epsilon)) * grad

    print(f"Step {t+1}: w = {w}")

Step 1: w = [7.68377223]
Step 2: w = [7.45878905]
Step 3: w = [7.27267368]
Step 4: w = [7.10900115]
Step 5: w = [6.960196]
Step 6: w = [6.82205318]
Step 7: w = [6.69196416]
Step 8: w = [6.56818559]
Step 9: w = [6.44948843]
Step 10: w = [6.33497062]
Step 11: w = [6.22394913]
Step 12: w = [6.11589368]
Step 13: w = [6.01038422]
Step 14: w = [5.90708238]
Step 15: w = [5.80571181]
Step 16: w = [5.70604412]
Step 17: w = [5.60788873]
Step 18: w = [5.51108525]
Step 19: w = [5.41549776]
Step 20: w = [5.32101038]


### 4.Adam (Adaptive Moment Estimation)

Adam combines the benefits of both Momentum and RMSProp. It uses moving averages of both the gradients (to build momentum) and their squared values (to adapt the learning rate), along with a bias correction mechanism to ensure updates are stable from the very first step.

In [ ]:
import numpy as np

W = np.random.randn(3, 1)
m = np.zeros_like(W)
v = np.zeros_like(W)
t = 0 # time step

learning_rate = 0.01
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8

# dummy gradient from backprop
dW = np.array([[0.2],
               [-0.1],
               [0.05]])

# Adam update
t += 1

m = beta1 * m + (1 - beta1) * dW
v = beta2 * v + (1 - beta2) * (dW ** 2)

m_hat = m / (1 - beta1**t)
v_hat = v / (1 - beta2**t)

W = W - learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)